### Energy Drink Consumer Survey Analysis - 
### Project Overview

This project analyzes consumer survey data to understand energy drink consumption behavior, brand preferences, purchasing patterns, health concerns, and price sensitivity.

The dataset contains demographic, lifestyle, consumption, and purchasing information collected from survey respondents. The primary objective is to identify meaningful patterns in consumer behavior and understand the factors that influence the price range of energy drinks purchased by consumers.

The project can also be extended into a Machine Learning classification problem, where price_range is used as the target variable to predict the price segment a consumer is likely to purchase. 

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import zscore
from sklearn.model_selection import train_test_split

In [2]:
df_survey = pd.read_csv('D:/Data Science Bootcamp 3.0/InterShip/project_resources1/project_resources1/survey_results.csv')
df_survey.head()

,respondent_id,age,gender,zone,occupation,income_levels,consume_frequency(weekly),current_brand,preferable_consumption_size,awareness_of_other_brands,reasons_for_choosing_brands,flavor_preference,purchase_channel,packaging_preference,health_concerns,typical_consumption_situations,price_range
0,R00001,30,M,Urban,Working Professional,<10L,3-4 times,Newcomer,Medium (500 ml),0 to 1,Price,Traditional,Online,Simple,Medium (Moderately health-conscious),"Active (eg. Sports, gym)",100-150
1,R00002,46,F,Metro,Working Professional,> 35L,5-7 times,Established,Medium (500 ml),2 to 4,Quality,Exotic,Retail Store,Premium,Medium (Moderately health-conscious),Social (eg. Parties),200-250
2,R00003,41,F,Rural,Working Professional,> 35L,3-4 times,Newcomer,Medium (500 ml),2 to 4,Availability,Traditional,Retail Store,Premium,Medium (Moderately health-conscious),"Active (eg. Sports, gym)",200-250
3,R00004,33,F,Urban,Working Professional,16L - 25L,5-7 times,Newcomer,Medium (500 ml),0 to 1,Brand Reputation,Exotic,Online,Eco-Friendly,Low (Not very concerned),"Active (eg. Sports, gym)",150-200
4,R00005,23,M,Metro,Student,NaN,3-4 times,Established,Medium (500 ml),0 to 1,Availability,Traditional,Online,Premium,Medium (Moderately health-conscious),"Active (eg. Sports, gym)",50-100


### Data Cleaning

### Ask - Identify duplicate and remove it

In [ ]:
# Remove duplicate - consider which columns help identify duplicate

# Check duplicates
print('Duplicate Count:- ',df_survey.duplicated().sum())

# duplicates = df_survey[df_survey.duplicated(subset=['respondent_id'], keep=False)]
duplicates = df_survey[df_survey.duplicated(subset=['respondent_id'])]
print('Duplicate Records')
duplicates

# Remove duplicate
df_survey.drop_duplicates(inplace=True)


Duplicate Count:-  10
Duplicate Records


### Outlier detection in Age

In [4]:
print('Max Age:- ', df_survey['age'].max())


# Calculate Z-score
df_survey['age_zscore'] = zscore(df_survey['age'])

#Identify outliers
outliers = df_survey[
    (df_survey['age_zscore'] < -3) |
    (df_survey['age_zscore'] > 3)
]
print('Length of Outlier:-',len(outliers))
print('Outlier')
outliers

# Remove outliers
df_survey_clean = df_survey[(df_survey['age_zscore'] >= -3) & (df_survey['age_zscore'] <= 3)]

# Remove temporary column
df_survey_clean = df_survey_clean.drop(columns=['age_zscore'])

print('Original Row:- ', len(df_survey))
print('Cleaned Row:-  ',len(df_survey_clean))

Max Age:-  604
Length of Outlier:- 9
Outlier
Original Row:-  30000
Cleaned Row:-   29991


### Ask - Handling mising data
1. For the income_levels column, missing values can be problematic. What would be a reasonable way to replace them? (Hint: Consider using “Not Reported” for missing income levels)
2. For the consume_frequency(weekly) and purchase_channel columns, think about the best way to fill missing values. Could the most common (mode) values be an appropriate replacement? Investigate the distributions before making a decision.

In [5]:
df_survey_clean['income_levels'] = df_survey_clean['income_levels'].fillna('Not Reported')

In [6]:
print(df_survey_clean['income_levels'].isna().sum())

0


In [7]:
# Both consume_frequency(weekly) and purchase_channel are categorical columns, so mode imputation can be appropriate. 
# But first, inspect their distributions and missing-value counts.

# Check missing values
print('Missing values for consume_frequency(weekly):- ', df_survey_clean['consume_frequency(weekly)'].isna().sum())
print('Missing valus for purchase_channel :- ',df_survey_clean['purchase_channel'].isna().sum())

# Distibution
print('Distribution of consume_frequency(weekly) :- ', df_survey_clean['consume_frequency(weekly)'].value_counts())
print('Distribution of purchase_channel :- ',df_survey_clean['purchase_channel'].value_counts())

Missing values for consume_frequency(weekly):-  8
Missing valus for purchase_channel :-  10
Distribution of consume_frequency(weekly) :-  consume_frequency(weekly)
3-4 times    11786
5-7 times     9774
0-2 times     8423
Name: count, dtype: int64
Distribution of purchase_channel :-  purchase_channel
Online          16562
Retail Store    13419
Name: count, dtype: int64


### Fill missing values with mode - Why mode - Both consume_frequency(weekly) and purchase_channel are categorical columns, so mode imputation can be appropriate

In [8]:
# Find the mode
print('Mode of consume_frequency(weekly):- ',df_survey_clean['consume_frequency(weekly)'].mode()[0])
print('Mode of purchase_channel:- ',df_survey_clean['purchase_channel'].mode()[0])


# Fill missing values with the mode
df_survey_clean['consume_frequency(weekly)'] = df_survey_clean['consume_frequency(weekly)'].fillna(df_survey_clean['consume_frequency(weekly)'].mode()[0])
df_survey_clean['purchase_channel'] = df_survey_clean['purchase_channel'].fillna(df_survey_clean['purchase_channel'].mode()[0])

# Vrify any null value
print('Any NULL after replcement of null values :- ',df_survey_clean[['consume_frequency(weekly)', 'purchase_channel']].isna().sum())

Mode of consume_frequency(weekly):-  3-4 times
Mode of purchase_channel:-  Online
Any NULL after replcement of null values :-  consume_frequency(weekly)    0
purchase_channel             0
dtype: int64


### Ask - Step4: Correcting Spelling Mistakes in Categorical Data:

Review the entries in the zone and current_brand columns. Are there any inconsistencies in spelling or formatting? How could you identify and fix these issues to ensure uniform categories?

In [9]:
## Correct data of zone column

#print(df_survey_clean['zone'].value_counts())

df_survey_clean['zone'] = df_survey_clean['zone'].replace({
    'Metor':'Metro',
    'urbna' :'Urban'
})

# Correct data of current_brand column
df_survey_clean['current_brand'] = df_survey_clean['current_brand'].replace('Establishd','Established')
df_survey_clean['current_brand'] = df_survey_clean['current_brand'].replace('newcomer','Newcomer')
df_survey_clean['current_brand'].value_counts()
#print(df_survey_clean['zone'].value_counts())

current_brand
Established    15462
Newcomer       14529
Name: count, dtype: int64

### Feature Engineering

In [10]:
#print(df_survey_clean.loc[df_survey_clean['age'] <= 25].value_counts())

print(df_survey_clean.loc[
    (df_survey_clean["age"] >= 46) &
    (df_survey_clean["age"] <= 55)
].value_counts())

respondent_id  age  gender  zone        occupation            income_levels  consume_frequency(weekly)  current_brand  preferable_consumption_size  awareness_of_other_brands  reasons_for_choosing_brands  flavor_preference  purchase_channel  packaging_preference  health_concerns                       typical_consumption_situations  price_range
R00002         46   F       Metro       Working Professional  > 35L          5-7 times                  Established    Medium (500 ml)              2 to 4                     Quality                      Exotic             Retail Store      Premium               Medium (Moderately health-conscious)  Social (eg. Parties)            200-250        1
R00010         49   F       Semi-Urban  Entrepreneur          10L - 15L      5-7 times                  Newcomer       Medium (500 ml)              2 to 4                     Price                        Traditional        Retail Store      Simple                High (Very health-conscious)          Acti

### Categorize Age into Age Groups

In [11]:
df_survey_clean["age_group"] = "Other"

df_survey_clean.loc[df_survey_clean['age'].between(18,25),'age_group']='18-25'
df_survey_clean.loc[df_survey_clean['age'].between(26,35),'age_group']='26-35'
df_survey_clean.loc[df_survey_clean['age'].between(36,45),'age_group']='36-45'
df_survey_clean.loc[df_survey_clean['age'].between(46,55),'age_group']='46-55'
df_survey_clean.loc[df_survey_clean['age'].between(56,70),'age_group']='56-70'
df_survey_clean.loc[df_survey_clean['age'] >= 70,'age_group']='70+'

df_survey_clean["age_group"].value_counts()

age_group
18-25    10468
26-35     9093
36-45     5972
46-55     2966
56-70     1398
70+         94
Name: count, dtype: int64

### Create `cf_ab_score` (Consume Frequency and Awareness Brand Score)

In [12]:
df_survey_clean['cf_ab_score']  = (
   np.where(df_survey_clean['consume_frequency(weekly)'] == '0-2 times',1,
   np.where(df_survey_clean['consume_frequency(weekly)'] == '3-4 times',2,
   np.where(df_survey_clean['consume_frequency(weekly)'] == '5-7 times',3,0)))
   /
   (   np.where(df_survey_clean["awareness_of_other_brands"] == "0 to 1", 1,
        np.where(df_survey_clean["awareness_of_other_brands"] == "2 to 4", 2,
        np.where(df_survey_clean["awareness_of_other_brands"] == "above 4", 3, 0)))
        +
        np.where(df_survey_clean["consume_frequency(weekly)"] == "0-2 times", 1,
        np.where(df_survey_clean["consume_frequency(weekly)"] == "3-4 times", 2,
        np.where(df_survey_clean["consume_frequency(weekly)"] == "5-7 times", 3, 0)))
    )
).round(2)

In [13]:
df_survey_clean[['respondent_id','consume_frequency(weekly)','awareness_of_other_brands','cf_ab_score']]

,respondent_id,consume_frequency(weekly),awareness_of_other_brands,cf_ab_score
0,R00001,3-4 times,0 to 1,0.67
1,R00002,5-7 times,2 to 4,0.60
2,R00003,3-4 times,2 to 4,0.50
3,R00004,5-7 times,0 to 1,0.75
4,R00005,3-4 times,0 to 1,0.67
...,...,...,...,...
30005,R29996,3-4 times,2 to 4,0.50
30006,R29997,3-4 times,2 to 4,0.50
30007,R29998,0-2 times,0 to 1,0.50
30008,R29999,5-7 times,above 4,0.50


In [14]:
print(df_survey_clean['cf_ab_score'].max())

0.75


### Step 3: Create Zone Affluence Score (ZAS)

In [15]:
df_survey_clean['zas_score']  = (
   np.where(df_survey_clean['zone'] == 'Urban',3,
   np.where(df_survey_clean['zone'] == 'Metro',4,
   np.where(df_survey_clean['zone'] == 'Rural',1,
   np.where(df_survey_clean['zone'] == 'Semi-Urban',2,0))))
   *
    np.where(df_survey_clean["income_levels"] == "<10L", 1,
    np.where(df_survey_clean["income_levels"] == "10L - 15L", 2,
    np.where(df_survey_clean["income_levels"] == "16L - 25L", 3,
    np.where(df_survey_clean["income_levels"] == "26L - 35L", 4,
    np.where(df_survey_clean["income_levels"] == "> 35L", 5,
    np.where(df_survey_clean["income_levels"] == "Not Reported", 0,0))))))       
    
).round(2)

In [16]:
df_survey_clean[['respondent_id','zone','income_levels','zas_score']]

 # tells how many unique values
print(df_survey_clean['zas_score'].nunique())

14


### Step 4: Brand Switching Indicator (BSI)

In [17]:
#df_survey_clean['bsi']  = np.where((df_survey_clean['current_brand'] != 'Established') & ((df_survey_clean['reasons_for_choosing_brands'] == 'Price') | df_survey_clean['reasons_for_choosing_brands'] == 'Quality'),1,0 )
df_survey_clean['bsi']  =np.where(
    (df_survey_clean['current_brand'] != 'Established') &
    (
        (df_survey_clean['reasons_for_choosing_brands'] == 'Price') |
        (df_survey_clean['reasons_for_choosing_brands'] == 'Quality')
    ),
    1,
    0
)

df_survey_clean[['respondent_id','current_brand','reasons_for_choosing_brands','bsi']]

#print('Condition Data')
df_survey_clean[df_survey_clean['bsi'] == 0]

#df_survey_clean.head()

,respondent_id,age,gender,zone,occupation,income_levels,consume_frequency(weekly),current_brand,preferable_consumption_size,awareness_of_other_brands,...,flavor_preference,purchase_channel,packaging_preference,health_concerns,typical_consumption_situations,price_range,age_group,cf_ab_score,zas_score,bsi
1,R00002,46,F,Metro,Working Professional,> 35L,5-7 times,Established,Medium (500 ml),2 to 4,...,Exotic,Retail Store,Premium,Medium (Moderately health-conscious),Social (eg. Parties),200-250,46-55,0.60,20,0
2,R00003,41,F,Rural,Working Professional,> 35L,3-4 times,Newcomer,Medium (500 ml),2 to 4,...,Traditional,Retail Store,Premium,Medium (Moderately health-conscious),"Active (eg. Sports, gym)",200-250,36-45,0.50,5,0
3,R00004,33,F,Urban,Working Professional,16L - 25L,5-7 times,Newcomer,Medium (500 ml),0 to 1,...,Exotic,Online,Eco-Friendly,Low (Not very concerned),"Active (eg. Sports, gym)",150-200,26-35,0.75,9,0
4,R00005,23,M,Metro,Student,Not Reported,3-4 times,Established,Medium (500 ml),0 to 1,...,Traditional,Online,Premium,Medium (Moderately health-conscious),"Active (eg. Sports, gym)",50-100,18-25,0.67,0,0
5,R00006,22,F,Urban,Student,Not Reported,5-7 times,Established,Large (1 L),2 to 4,...,Traditional,Online,Simple,Low (Not very concerned),"Active (eg. Sports, gym)",100-150,18-25,0.60,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30004,R29995,67,M,Semi-Urban,Retired,<10L,5-7 times,Newcomer,Small (250 ml),2 to 4,...,Exotic,Retail Store,Premium,Medium (Moderately health-conscious),"Active (eg. Sports, gym)",150-200,56-70,0.60,2,0
30005,R29996,29,M,Metro,Working Professional,16L - 25L,3-4 times,Newcomer,Medium (500 ml),2 to 4,...,Traditional,Online,Simple,High (Very health-conscious),Social (eg. Parties),150-200,26-35,0.50,12,0
30006,R29997,42,F,Urban,Working Professional,> 35L,3-4 times,Established,Small (250 ml),2 to 4,...,Traditional,Online,Premium,Low (Not very concerned),"Active (eg. Sports, gym)",200-250,36-45,0.50,15,0
30007,R29998,57,F,Metro,Retired,<10L,0-2 times,Established,Medium (500 ml),0 to 1,...,Traditional,Retail Store,Premium,High (Very health-conscious),Casual (eg. At home),150-200,56-70,0.50,4,0


### Removing Logical Outliers:

In [18]:
# For instance, there are students listed in the `56-70` age group, which seems like an
# incorrect entry

df_survey_clean = df_survey_clean[
    ~(
        (df_survey_clean["occupation"] == "Student") &
        (df_survey_clean["age_group"].isin(["56-70", "70+"]))
    )
]
df_survey_clean.shape[0]

29956

### Ask - How many rows contain the Brand Switching Indicator (BSI) with a value of 0 after removing logical outliers?

In [19]:
bsi_zero_count  =(df_survey_clean["bsi"] == 0).sum()
print(bsi_zero_count)

20796


### Modeling

### Prepare features and Traget variables

In [21]:
df_survey_clean.columns

Index(['respondent_id', 'age', 'gender', 'zone', 'occupation', 'income_levels',
       'consume_frequency(weekly)', 'current_brand',
       'preferable_consumption_size', 'awareness_of_other_brands',
       'reasons_for_choosing_brands', 'flavor_preference', 'purchase_channel',
       'packaging_preference', 'health_concerns',
       'typical_consumption_situations', 'price_range', 'age_group',
       'cf_ab_score', 'zas_score', 'bsi'],
      dtype='str')

In [20]:
df_modeling = df_survey_clean.copy()

In [21]:
print(df_modeling.info())
#df.info()

<class 'pandas.DataFrame'>
Index: 29956 entries, 0 to 30009
Data columns (total 21 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   respondent_id                   29956 non-null  str    
 1   age                             29956 non-null  int64  
 2   gender                          29956 non-null  str    
 3   zone                            29956 non-null  str    
 4   occupation                      29956 non-null  str    
 5   income_levels                   29956 non-null  str    
 6   consume_frequency(weekly)       29956 non-null  str    
 7   current_brand                   29956 non-null  str    
 8   preferable_consumption_size     29956 non-null  str    
 9   awareness_of_other_brands       29956 non-null  str    
 10  reasons_for_choosing_brands     29956 non-null  str    
 11  flavor_preference               29956 non-null  str    
 12  purchase_channel                29956 non-null  

### Feature Encoding

In [22]:
# Apply appropriate encoding techniques to the features:
# Use Label Encoding for the following columns:
# age_group
# income_levels
# health_concerns
# consume_frequency (weekly)
# preferable_consumption_size

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df_modeling['age_group'] = le.fit_transform(df_modeling['age_group'])
df_modeling['income_levels'] = le.fit_transform(df_modeling['income_levels'])
df_modeling['health_concerns'] = le.fit_transform(df_modeling['health_concerns'])
df_modeling['consume_frequency(weekly)'] = le.fit_transform(df_modeling['consume_frequency(weekly)'])
df_modeling['preferable_consumption_size'] = le.fit_transform(df_modeling['preferable_consumption_size'])
df_modeling['price_range'] = le.fit_transform(df_modeling['price_range'])
#df_modeling['income_levels'].unique()
# Dropping age column
df_modeling = df_modeling.drop(columns=['age'])
df_modeling.head()
#df_modeling.columns



,respondent_id,gender,zone,occupation,income_levels,consume_frequency(weekly),current_brand,preferable_consumption_size,awareness_of_other_brands,reasons_for_choosing_brands,flavor_preference,purchase_channel,packaging_preference,health_concerns,typical_consumption_situations,price_range,age_group,cf_ab_score,zas_score,bsi
0,R00001,M,Urban,Working Professional,3,1,Newcomer,1,0 to 1,Price,Traditional,Online,Simple,2,"Active (eg. Sports, gym)",0,1,0.67,3,1
1,R00002,F,Metro,Working Professional,4,2,Established,1,2 to 4,Quality,Exotic,Retail Store,Premium,2,Social (eg. Parties),2,3,0.60,20,0
2,R00003,F,Rural,Working Professional,4,1,Newcomer,1,2 to 4,Availability,Traditional,Retail Store,Premium,2,"Active (eg. Sports, gym)",2,2,0.50,5,0
3,R00004,F,Urban,Working Professional,1,2,Newcomer,1,0 to 1,Brand Reputation,Exotic,Online,Eco-Friendly,1,"Active (eg. Sports, gym)",1,1,0.75,9,0
4,R00005,M,Metro,Student,5,1,Established,1,0 to 1,Availability,Traditional,Online,Premium,2,"Active (eg. Sports, gym)",3,0,0.67,0,0


In [23]:
# One-Hot coding

from sklearn.preprocessing import OneHotEncoder

#df_modeling['zone']
# encoder = OneHotEncoder()
# encoder.fit_transform(df_modeling)
df_modeling = pd.get_dummies(df_modeling,columns=['gender'],drop_first=True,dtype=int)
df_modeling = pd.get_dummies(df_modeling,columns=['zone'],drop_first=True,dtype=int)
df_modeling = pd.get_dummies(df_modeling,columns=['occupation'],drop_first=True,dtype=int)
df_modeling = pd.get_dummies(df_modeling,columns=['current_brand'],drop_first=True,dtype=int)
df_modeling = pd.get_dummies(df_modeling,columns=['awareness_of_other_brands'],drop_first=True,dtype=int)
df_modeling = pd.get_dummies(df_modeling,columns=['reasons_for_choosing_brands'],drop_first=True,dtype=int)
df_modeling = pd.get_dummies(df_modeling,columns=['flavor_preference'],drop_first=True,dtype=int)
df_modeling = pd.get_dummies(df_modeling,columns=['purchase_channel'],drop_first=True,dtype=int)
df_modeling = pd.get_dummies(df_modeling,columns=['typical_consumption_situations'],drop_first=True,dtype=int)
df_modeling = pd.get_dummies(df_modeling,columns=['packaging_preference'],drop_first=True,dtype=int)
df_modeling.head()
#df_modeling['bsi']

,respondent_id,income_levels,consume_frequency(weekly),preferable_consumption_size,health_concerns,price_range,age_group,cf_ab_score,zas_score,bsi,...,awareness_of_other_brands_above 4,reasons_for_choosing_brands_Brand Reputation,reasons_for_choosing_brands_Price,reasons_for_choosing_brands_Quality,flavor_preference_Traditional,purchase_channel_Retail Store,typical_consumption_situations_Casual (eg. At home),typical_consumption_situations_Social (eg. Parties),packaging_preference_Premium,packaging_preference_Simple
0,R00001,3,1,1,2,0,1,0.67,3,1,...,0,0,1,0,1,0,0,0,0,1
1,R00002,4,2,1,2,2,3,0.60,20,0,...,0,0,0,1,0,1,0,1,1,0
2,R00003,4,1,1,2,2,2,0.50,5,0,...,0,0,0,0,1,1,0,0,1,0
3,R00004,1,2,1,1,1,1,0.75,9,0,...,0,1,0,0,0,0,0,0,0,0
4,R00005,5,1,1,2,3,0,0.67,0,0,...,0,0,0,0,1,0,0,0,1,0


In [26]:
#df_modeling = df_modeling.drop(columns=['respondent_id'])
#df_modeling.head()


### Data Spliting

In [25]:
X = df_modeling.drop(['respondent_id','price_range'], axis=1)
y = df_modeling['price_range']

#  Split the data into training and test sets with a test size of 25%
X_train, X_test,y_train, y_test = train_test_split(X,y,test_size=0.25,random_state=42)

### Model Training Using  Logistic Regression

In [26]:
# Model - Logistic Regression.

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

#solver='lbfgs' --> It is the optimization algorithm used to train Logistic Regression.
params = { 'solver':'lbfgs'}
model_logistic = LogisticRegression(solver='lbfgs')
model_logistic.fit(X_train,y_train)

# Make the prediction on test set using trained model.
y_pred = model_logistic.predict(X_test)

# Evaluate the model using classification report.
logistic_classi_report = classification_report(y_test,y_pred)
print(f"Classification Report:-")
print(logistic_classi_report)
logistic_accuracy_score = accuracy_score(y_test, y_pred)
print(' Logistic Regression Accuracy Report:-',round(logistic_accuracy_score, 2))

Classification Report:-
              precision    recall  f1-score   support

           0       0.73      0.76      0.74      1930
           1       0.75      0.77      0.76      2223
           2       0.91      0.89      0.90      2430
           3       0.79      0.73      0.76       906

    accuracy                           0.80      7489
   macro avg       0.80      0.79      0.79      7489
weighted avg       0.80      0.80      0.80      7489

 Logistic Regression Accuracy Report:- 0.8


c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


### Model Training Using RandomForestClassifier

In [27]:
from sklearn.ensemble import RandomForestClassifier

# n_estimators=30, max_depth=3

params = { 'n_estimators':30,'max_depth':3}
model_randomforest = RandomForestClassifier(n_estimators=30,max_depth =3)
model_randomforest.fit(X_train,y_train)
y_pred_r = model_randomforest.predict(X_test)

randomforest_report = classification_report(y_test,y_pred_r)
print(f'Random Forest Report :- ')
print(randomforest_report)
randomforest_accuracy_score = accuracy_score(y_test, y_pred_r)
print(' Logistic Regression Accuracy Report:-',round(randomforest_accuracy_score, 2))

Random Forest Report :- 
              precision    recall  f1-score   support

           0       0.59      0.54      0.56      1930
           1       0.54      0.67      0.60      2223
           2       0.76      0.83      0.79      2430
           3       0.99      0.31      0.47       906

    accuracy                           0.65      7489
   macro avg       0.72      0.59      0.61      7489
weighted avg       0.68      0.65      0.64      7489

 Logistic Regression Accuracy Report:- 0.65


### Model Training Using SVM: RBF Kernel

In [28]:
# We use SVM when ->  Good general-purpose choice
from sklearn.svm import SVC
params = { 'kernel':'rbf'}
model_svm = SVC(kernel='rbf')

model_svm.fit(X_train,y_train)
y_pred_s = model_svm.predict(X_test)

svm_report = classification_report(y_test,y_pred_s)
print(f'SVM Report :- ')
print(svm_report)
svm_accuracy_score = accuracy_score(y_test, y_pred_s)
print('SVM Accuracy Report:-',round(svm_accuracy_score, 2))

SVM Report :- 
              precision    recall  f1-score   support

           0       0.76      0.79      0.78      1930
           1       0.78      0.80      0.79      2223
           2       0.91      0.91      0.91      2430
           3       0.83      0.73      0.78       906

    accuracy                           0.82      7489
   macro avg       0.82      0.81      0.81      7489
weighted avg       0.83      0.82      0.82      7489

SVM Accuracy Report:- 0.82


### Model Training Using XGBoost

In [29]:
from xgboost import XGBClassifier
# Why eval_metric='logloss' --> Without this parameter model prediction is not confident.
# eval_metric = Performance measure during training
# 'logloss' = Measures probability prediction quality
# Your project has multiple classes, so logloss is a suitable evaluation metric.
params = { 'eval_metric':'logloss'}

model_xgb = XGBClassifier(eval_metric='logloss')
model_xgb.fit(X_train,y_train)
y_pred_xgb = model_xgb.predict(X_test)

# Print classification report
xgb_report = classification_report(y_test,y_pred_xgb)
print(f'XGBoost Report :- ')
print(xgb_report)
xgb_accuracy_score = accuracy_score(y_test, y_pred_xgb)
print('XGB Accuracy Report:-',round(xgb_accuracy_score, 2))

XGBoost Report :- 
              precision    recall  f1-score   support

           0       0.92      0.90      0.91      1930
           1       0.89      0.91      0.90      2223
           2       0.95      0.95      0.95      2430
           3       0.92      0.92      0.92       906

    accuracy                           0.92      7489
   macro avg       0.92      0.92      0.92      7489
weighted avg       0.92      0.92      0.92      7489

XGB Accuracy Report:- 0.92


### Model Training Using Light Gradient Boosting Machine (Light GBM)
%pip install lightgbm

In [30]:
from lightgbm import LGBMClassifier

params = { 'n_estimators':100,'learning_rate':0.1,'random_state':42}

model_lgbm = LGBMClassifier(n_estimators=100,learning_rate=0.1,random_state=4)
model_lgbm.fit(X_train,y_train)
y_pred_lgbm = model_lgbm.predict(X_test)
lgbm_report = classification_report(y_test,y_pred_lgbm)

print('LGBM Classficiation Report:- ')
print(lgbm_report)
lgbm_accuracy_score = accuracy_score(y_test, y_pred_lgbm)
print('Accuracy Report:-',round(lgbm_accuracy_score, 2))



[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001680 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 83
[LightGBM] [Info] Number of data points in the train set: 22467, number of used features: 27
[LightGBM] [Info] Start training from score -1.343386
[LightGBM] [Info] Start training from score -1.228925
[LightGBM] [Info] Start training from score -1.126779
[LightGBM] [Info] Start training from score -2.100810
LGBM Classficiation Report:- 
              precision    recall  f1-score   support

           0       0.91      0.90      0.90      1930
           1       0.89      0.91      0.90      2223
           2       0.96      0.95      0.96      2430
           3       0.92      0.92      0.92       906

    accuracy                           0.92      74

### Model Training Using Gaussian Naive Bayes

In [31]:
#Step 1: Initialize and train a Gaussian Naive Bayes model using the training data
from sklearn.naive_bayes import GaussianNB
params = {'var_smoothing':1e-9}
model_gnb = GaussianNB(var_smoothing=1e-9)
model_gnb.fit(X_train,y_train)

# Step 2: Make predictions on the test set using the trained model
y_pread_gnb = model_gnb.predict(X_test)

report_gnb = classification_report(y_test,y_pread_gnb)

print('Gaussian Naive Bayes Classficiation Report:- ')
print(report_gnb)
accuracy_score_gnb = accuracy_score(y_test, y_pread_gnb)
print('Gaussian Naive Bayes Accuracy Report:-',round(accuracy_score_gnb, 2))

Gaussian Naive Bayes Classficiation Report:- 
              precision    recall  f1-score   support

           0       0.48      0.29      0.36      1930
           1       0.56      0.31      0.40      2223
           2       0.66      0.88      0.75      2430
           3       0.45      0.91      0.60       906

    accuracy                           0.56      7489
   macro avg       0.54      0.60      0.53      7489
weighted avg       0.56      0.56      0.53      7489

Gaussian Naive Bayes Accuracy Report:- 0.56


### ML Flow Installation

In [32]:
import mlflow

print(mlflow.__version__)

3.16.1


In [33]:
models = [
    (
        "Gaussian Naive Bayes", 
        GaussianNB(var_smoothing=1e-9),
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "Logistic Regression", 
        LogisticRegression(solver='lbfgs'),
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "Suport Vector Machine",
        SVC(kernel='rbf'), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "Random Forest",
        RandomForestClassifier(n_estimators=30,max_depth =3), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBoost",
        XGBClassifier(eval_metric='logloss'), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "Light Gradient Boosting Machine",
        LGBMClassifier(n_estimators=100,learning_rate=0.1,random_state=4), 
        (X_train, y_train),
        (X_test, y_test)
        )
]

In [34]:
reports = []

for model_name, model, train_set, test_set in models:
    X_train = train_set[0]
    y_train = train_set[1]
    X_test = test_set[0]
    y_test = test_set[1]
    
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    report = classification_report(y_test, y_pred, output_dict=True)
    reports.append(report)

c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002445 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 83
[LightGBM] [Info] Number of data points in the train set: 22467, number of used features: 27
[LightGBM] [Info] Start training from score -1.343386
[LightGBM] [Info] Start training from score -1.228925
[LightGBM] [Info] Start training from score -1.126779
[LightGBM] [Info] Start training from score -2.100810


In [35]:
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import mlflow
import mlflow.sklearn
import mlflow.xgboost
import mlflow.lightgbm


In [37]:
# Initialize MLflow
mlflow.set_experiment("Energey Drink Survey Base on Price Range")
mlflow.set_tracking_uri("http://127.0.0.1:5000/")

for i, element in enumerate(models):
    model_name = element[0]
    model = element[1]
    report = reports[i]
    
    with mlflow.start_run(run_name=model_name):        
        mlflow.log_param("model", model_name)
        mlflow.log_metric('accuracy', report['accuracy'])
        mlflow.log_metric('recall_class_0', report['0']['recall'])
        mlflow.log_metric('recall_class_1', report['1']['recall'])
        mlflow.log_metric('recall_class_2', report['2']['recall'])
        mlflow.log_metric('recall_class_3', report['3']['recall'])
        mlflow.log_metric('f1_score_macro', report['macro avg']['f1-score'])        
        
        if "XGB" in model_name:
            mlflow.xgboost.log_model(model, "model")
        elif "Light Gradient" in model_name:
            mlflow.lightgbm.log_model(model, "model")           
        else:
            mlflow.sklearn.log_model(model, "model",skops_trusted_types=["sklearn.tree._tree.Tree"])

2026/09/30 08:13:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Gaussian Naive Bayes at: http://127.0.0.1:5000/#/experiments/2/runs/3bc9abcfa9fe4b4ab9b3e1aa6aba0fcc
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


2026/09/30 08:14:41 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Logistic Regression at: http://127.0.0.1:5000/#/experiments/2/runs/f5c082732ac14d9faa0a8aa01e0e8ecb
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


2026/09/30 08:14:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


...

🏃 View run Random Forest at: http://127.0.0.1:5000/#/experiments/2/runs/a67f40b325894c158333071d190e549e
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


2026/09/30 08:15:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBoost at: http://127.0.0.1:5000/#/experiments/2/runs/d243c06de74b4b1795fec218159c1950
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


2026/09/30 08:15:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Light Gradient Boosting Machine at: http://127.0.0.1:5000/#/experiments/2/runs/d592e876cd8c4715894588671ca43cb9
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


### Register the Model

In [ ]:
model_name = "XGBoost"

run_id = input("Please type RunID: ")

model_uri = f"runs:/{run_id}/model"

mlflow.register_model(
    model_uri=model_uri,
    name=model_name
)

Registered model 'XGBoost' already exists. Creating a new version of this model...
2026/09/30 08:17:19 WARNING mlflow.tracking._model_registry.fluent: Run with id d243c06de74b4b1795fec218159c1950 has no artifacts at artifact path 'model', registering model based on models:/m-3ee57d2b39f344bfa5a2eb567ee94e89 instead
2026/09/30 08:17:20 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGBoost, version 2
Created version '2' of model 'XGBoost'.


<ModelVersion: aliases=[], creation_timestamp=1790736440065, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1790736440065, metrics=None, model_id=None, name='XGBoost', params=None, run_id='d243c06de74b4b1795fec218159c1950', run_link='', source='models:/m-3ee57d2b39f344bfa5a2eb567ee94e89', status='READY', status_message=None, tags={}, user_id='', version='2', workspace='default'>